# Fundamentals 08 - System API

Objetivo: mostrar `AgenticSystem` como workspace y fábrica de composición. Un system no es un agente: registra tools, skills, agents, runtime y contratos para que la ejecucion sea auditable y repetible.

Modelo mental:

```text
System = registry + runtime + skills + agents + inspect
```


In [ ]:
import agentic_systems as lab

PRETTY = False

scheduler = lab.scheduler(timeout_s=30, max_retries=0, max_tool_calls=8, max_turns=8)
runtime = lab.runtime(provider="python-direct", model="local-python", region="local", scheduler=scheduler)

system = lab.AgenticSystem(model="local-python", region="local", runtime=runtime)

## Escenario did?ctico compartido

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```


In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Escenario did?ctico")

## 1) Registrar tools en el system

`system.tool(...)` mete la tool en el registry del workspace. Esto sirve cuando varias skills o agentes deben compartir la misma capacidad sin duplicar definiciones.

In [ ]:
@system.tool
def sumar(a: int, b: int) -> dict:
    return {"operation": "sumar", "result": a + b, "explanation": f"{a} + {b} = {a + b}"}


@system.tool
def restar(a: int, b: int) -> dict:
    return {"operation": "restar", "result": a - b, "explanation": f"{a} - {b} = {a - b}"}


@system.tool
def multiplicar(a: int, b: int) -> dict:
    return {"operation": "multiplicar", "result": a * b, "explanation": f"{a} * {b} = {a * b}"}


@system.tool
def dividir(a: float, b: float) -> dict:
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    return {"operation": "dividir", "result": value, "explanation": f"{a} / {b} = {value}"}

lab.show({
    "tool_names": list(system.public_tool_names),
    "runtime_tool_names": list(system.tool_names),
}, title="System registry")

## 2) Empaquetar esas tools como skill runtime

La skill da nombre, prompts, contratos y policy a un paquete de tools. El system la puede registrar y expandir en agentes sin perder el contrato.

In [ ]:
calculator_contract = lab.AgentContract(
    must_call=["sumar", "restar", "multiplicar", "dividir"],
    completion="when_required_tools_satisfied",
)
calculator_policy = lab.RunPolicy(max_tool_calls=4, max_turns=4)

math_skill = lab.Skill(
    name="system_math",
    description="Skill aritmetica compartida por el system.",
    tools=[system.public_tools["sumar"], system.public_tools["restar"], system.public_tools["multiplicar"], system.public_tools["dividir"]],
    prompts={"instructions": "Usa tools aritmeticas y conserva evidencia estructurada."},
    contracts={"default": calculator_contract.model_dump(mode="json")},
    policy=calculator_policy.model_dump(mode="json"),
)

system.skill(math_skill)

lab.show({
    "skill_names": list(system.skill_names),
    "runtime_skills": [skill.info() for skill in system.runtime_skills],
}, title="System skills")

## 3) Crear un agente desde el system

`system.agent(...)` resuelve tools y skills desde el workspace. El agente queda ligado al runtime del system y puede ejecutarse con `python-direct` para pruebas deterministas.

In [ ]:
single_step_contract = lab.AgentContract(must_call=["sumar"], completion="when_required_tools_satisfied")

calculator_agent = system.agent(
    name="system_calculator_agent",
    instructions=math_skill.instructions,
    skills=[math_skill],
    engine="python-direct",
    contract=single_step_contract,
    policy=lab.RunPolicy(max_tool_calls=1, max_turns=1),
    runtime=runtime,
)

result = calculator_agent.run({"tool": "sumar", "input": {"a": 10, "b": 20}}, mode="eval")

lab.human_result(
    result,
    title="Human result - system.agent(...)",
    expected_tools=lab.expect.exactly("sumar"),
    pretty=PRETTY,
)

## 4) Pipeline determinista dentro del system

Un pipeline no necesita un loop reactivo. Puede ejecutar tools registradas en orden fijo, conservar evidencia y producir un resultado auditable con `compose_result`.

In [ ]:
operation_plan = [
    ("sumar", {"a": 10, "b": 20}),
    ("restar", {"a": 30, "b": 9}),
    ("multiplicar", {"a": 21, "b": 4}),
    ("dividir", {"a": 84, "b": 2}),
]

operation_trace = []
tool_results = []
for tool_name, tool_input in operation_plan:
    tool_result = system.public_tools[tool_name].run(tool_input)
    operation_trace.append(tool_result.data)
    tool_results.append(tool_result)

pipeline_final = {
    "procedimiento": [item["explanation"] for item in operation_trace],
    "resultado_final": operation_trace[-1]["result"],
}

pipeline_result = lab.compose_result(
    text="El pipeline determinista resolvio el escenario did?ctico.",
    data=pipeline_final,
    results=tool_results,
    mode="pipeline",
    input=USER_PROMPT,
    meta={"system": "fundamentals", "operation_trace": operation_trace},
)

lab.human_result(
    pipeline_result,
    title="Human result - deterministic system pipeline",
    pretty=PRETTY,
)

## 5) Inspeccionar el workspace

`system.inspect()` resume tools, agents, skills y errores de contrato. Es la auditoria local antes de llevar el mismo sistema a un runtime cloud.

In [ ]:
inspection = system.inspect()
inspection.raise_if_errors()
lab.show(inspection, title="System inspect")

## Lo importante

- `AgenticSystem` es el workspace: registra tools, skills y agents.
- `system.agent(...)` crea agentes ligados al runtime y al registry del system.
- Un pipeline determinista no necesita `loop=...`; es una composicion explicita de tools.
- `system.inspect()` valida el workspace antes de usarlo como base de graphs o environments.

## Coverage API de este notebook

Esta tabla deja explicito que parte de Agentic Systems queda materializada aqui.

In [ ]:
api_coverage = [
    {"api": "lab.AgenticSystem", "description": "Crea el workspace y fábrica de tools, skills, agents y runtime."},
    {"api": "system.tool", "description": "Registra tools reutilizables en el registry del system."},
    {"api": "system.skill", "description": "Registra una Skill runtime y expande sus tools."},
    {"api": "system.agent", "description": "Crea agentes ligados al registry y runtime del system."},
    {"api": "deterministic pipeline", "description": "Ejecuta tools en orden fijo sin introducir un parametro loop nuevo."},
    {"api": "system.inspect", "description": "Audita el workspace antes de componer graphs o environments."},
]

lab.show({"notebook": "08_system_api.ipynb", "api_coverage": api_coverage})

## S?mbolos API explicados

Este notebook se alinea con `docs/API.md` y ense?a estos s?mbolos p?blicos:

- `AgenticSystem`: Workspace nativo para tools, skills, agentes y pipelines.
- `lab.compose_result`: Resultado compuesto para pipelines deterministas.
- `core / providers / integrations`: Namespaces p?blicos que delimitan capas.
- `Skill`: Skill registrada dentro del system.
- `human_result`: Render del resultado del system.

